# Sentinel-2 monthly composites for Kakamega, Kilifi and Nyeri (2020-2023, 2025-2026)

**Method**
1. Drop scenes with `CLOUDY_PIXEL_PERCENTAGE > 60`, then mask remaining cloudy pixels with Cloud Score+.
2. Build a monthly median composite.
3. Fill gaps from nearby dates: first a ±1 month window, then ±2 months.
4. Add a `fill_flag` band: `0` = original month, `1` = filled from ±1 month, `2` = filled from ±2 months, `255` = no valid data in any window (spectral bands are `0` there).
5. Clip to each county and export one multiband GeoTIFF per county per month. Earth Engine can only export to Google Drive, so the files land in Drive first.
6. A live monitor shows export progress and moves finished files from Drive (via Google Drive for desktop) to `F:\s2_bands`, with progress bars.
7. (Optional) Stack the monthly files per county in Python.

**Before you start:** install *Google Drive for desktop* so your Drive shows up as a drive letter (usually `G:`). Free Drive space is only 15 GB, so moving files out as they finish (this notebook does it) and emptying the Drive trash matters.

In [1]:
# Run once
%pip install -q earthengine-api geemap geopandas rioxarray xarray dask tqdm

Note: you may need to restart the kernel to use updated packages.


In [1]:
import ee, geemap, geopandas as gpd

ee.Authenticate()   # first time only; opens a browser login
ee.Initialize(project="tonn-512407")   # <-- put your Google Cloud / GEE project ID here

## 1. Settings

In [ ]:
COUNTY_SHP = r"C:\Users\locateIT1\Downloads\ken_admin1.shp"   # national admin-1 shapefile
COUNTY_NAME_COL = "adm1_name"
COUNTIES = ["Kakamega", "Kilifi", "Nyeri"]

YEARS = [2020, 2021, 2022, 2023, 2025, 2026]   # 2024 is skipped
LAST_MONTH_2026 = 9                            # 2026 is only exported up to this month

MAX_CLOUD = 60            # scene-level cloud cover limit (%)
CS_THRESHOLD = 0.60       # Cloud Score+ pixel clarity (higher = stricter masking)
FILL_WINDOWS = [1, 2]     # months either side used for gap filling
SCALE = 10                # metres; use 20 to make files ~4x smaller
DRIVE_FOLDER = "S2_monthly_counties"

BANDS = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]

# UTM zone per county
EXPORT_CRS = {"Kakamega": "EPSG:32636", "Kilifi": "EPSG:32737", "Nyeri": "EPSG:32737"}

# Where files end up
LOCAL_DIR = r"F:\s2_bands"                                     # final destination
MOVE_FILES = True                                              # False = only monitor exports
DRIVE_SYNC_DIR = r"G:\My Drive\S2_monthly_counties"            # Drive for desktop path; adjust if different

## 2. Load counties from the shapefile

In [ ]:
gdf = gpd.read_file(COUNTY_SHP).to_crs(4326)
gdf = gdf[gdf[COUNTY_NAME_COL].str.lower().isin([c.lower() for c in COUNTIES])].copy()
print("Matched counties:", gdf[COUNTY_NAME_COL].tolist())
assert len(gdf) == len(COUNTIES), "Not all counties matched - check names in the shapefile"

# Standardise names to the ones in COUNTIES
name_map = {c.lower(): c for c in COUNTIES}
gdf[COUNTY_NAME_COL] = gdf[COUNTY_NAME_COL].str.lower().map(name_map)

county_geoms = {}
for name in COUNTIES:
    sub = gdf[gdf[COUNTY_NAME_COL] == name]
    county_geoms[name] = geemap.gdf_to_ee(sub).geometry()

all_aoi = geemap.gdf_to_ee(gdf).geometry()

## 3. Sentinel-2 collection, cloud filtering and masking

In [ ]:
s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(all_aoi)
      .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", MAX_CLOUD)))

cs_plus = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
s2 = s2.linkCollection(cs_plus, ["cs_cdf"])

def mask_clouds(img):
    return img.updateMask(img.select("cs_cdf").gte(CS_THRESHOLD)).select(BANDS)

s2 = s2.map(mask_clouds)

## 4. Monthly composite with gap filling from nearby months

In [ ]:
EMPTY = ee.Image.constant([0] * len(BANDS)).rename(BANDS).toUint16().updateMask(0)

def composite(start, end):
    col = s2.filterDate(start, end)
    # if a window has no scenes at all, return a fully masked image instead of failing
    return ee.Image(ee.Algorithms.If(col.size().gt(0), col.median(), EMPTY))

def build_month(y, m):
    start = ee.Date.fromYMD(y, m, 1)
    base = composite(start, start.advance(1, "month"))

    filled = base
    # 255 = still empty, 0 = original month, w = filled from +/- w months
    flag = ee.Image(255).where(base.select("B2").mask(), 0)

    for w in FILL_WINDOWS:
        win = composite(start.advance(-w, "month"), start.advance(w + 1, "month"))
        newly = flag.eq(255).And(win.select("B2").mask())
        flag = flag.where(newly, w)
        filled = filled.unmask(win)

    # Export turns masked pixels into 0, so keep the flag unmasked (255 = no data)
    # and write 0 explicitly into the spectral bands where nothing was found.
    return (filled.unmask(0).toInt16()
            .addBands(flag.rename("fill_flag").toInt16()))

## 5. (Optional) Quick test on one month
Check the result on a map before starting the exports.

In [ ]:
test = build_month(2023, 3)
valid = test.select("fill_flag").lt(255)
Map = geemap.Map()
Map.centerObject(all_aoi, 7)
Map.addLayer(test.updateMask(valid), {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}, "RGB Mar 2023")
Map.addLayer(test.select("fill_flag").updateMask(valid), {"min": 0, "max": 2, "palette": ["white", "orange", "red"]}, "fill_flag")
Map.addLayer(all_aoi, {}, "Counties")
Map

## 6. Start export tasks
One GeoTIFF per county per month, sent to Google Drive (69 months × 3 counties = 207 tasks).
A progress bar shows the submission. Tasks you already submitted (queued, running or completed) are skipped, so re-running this cell will not create duplicates.

In [ ]:
from tqdm.auto import tqdm

months = [(y, m) for y in YEARS for m in range(1, 13)
          if not (y == 2026 and m > LAST_MONTH_2026)]
print(f"{len(months)} months x {len(COUNTIES)} counties = {len(months) * len(COUNTIES)} tasks")

# descriptions of tasks that already exist, so we don't resubmit them
existing = {x.config.get("description") for x in ee.batch.Task.list()
            if x.state in ("READY", "RUNNING", "COMPLETED")}

submitted, skipped = 0, 0
for (y, m) in tqdm(months, desc="Submitting exports", unit="month"):
    img = None
    for name in COUNTIES:
        desc = f"S2_{name}_{y}_{m:02d}"
        if desc in existing:
            skipped += 1
            continue
        if img is None:
            img = build_month(y, m)
        geom = county_geoms[name]
        t = ee.batch.Export.image.toDrive(
            image=img.clip(geom),
            description=desc,
            folder=DRIVE_FOLDER,
            fileNamePrefix=desc,
            region=geom,
            scale=SCALE,
            crs=EXPORT_CRS[name],
            maxPixels=1e10,
            fileFormat="GeoTIFF",
        )
        t.start()
        submitted += 1

print(f"Submitted {submitted} new tasks, skipped {skipped} that already existed.")

## 7. Live progress: exports and transfer to F:\s2_bands
Run this cell and leave it running. It refreshes every 30 seconds and shows:
- a progress bar for the exports running on Earth Engine, with counts of completed, running, queued and failed tasks;
- a progress bar for files saved to `F:\s2_bands` (finished exports are moved out of Drive automatically, which also frees Drive space).

Stopping the cell does **not** cancel the exports (they run on Google's servers). Re-run the cell any time to pick up where it left off. Remember to empty the Drive trash now and then, since trashed files still count toward your quota.

In [ ]:
import os, glob, shutil, time
from collections import Counter
from IPython.display import clear_output

expected = {f"S2_{name}_{y}_{m:02d}" for (y, m) in months for name in COUNTIES}

def bar(done, total, width=40):
    filled = int(width * done / total) if total else 0
    return f"[{'█' * filled}{'░' * (width - filled)}] {done}/{total}"

def latest_tasks():
    seen = {}
    for x in ee.batch.Task.list():          # newest first
        d = x.config.get("description")
        if d in expected and d not in seen:
            seen[d] = x
    return seen

def run(poll_seconds=30, move_files=MOVE_FILES):
    total = len(expected)
    os.makedirs(LOCAL_DIR, exist_ok=True)
    if move_files and not os.path.isdir(DRIVE_SYNC_DIR):
        print(f"WARNING: {DRIVE_SYNC_DIR} not found. Install Google Drive for desktop or fix DRIVE_SYNC_DIR. "
              "Only monitoring exports for now.")
        move_files = False

    moved = {d for d in expected if glob.glob(os.path.join(LOCAL_DIR, d + "*.tif"))}
    last_file = "-"

    while True:
        seen = latest_tasks()
        states = Counter(x.state for x in seen.values())
        completed = {d for d, x in seen.items() if x.state == "COMPLETED"}
        failed = {d for d, x in seen.items() if x.state in ("FAILED", "CANCELLED", "CANCEL_REQUESTED")}

        if move_files:
            for d in sorted(completed - moved):
                for f in glob.glob(os.path.join(DRIVE_SYNC_DIR, d + "*.tif")):
                    print(f"moving {os.path.basename(f)} to {LOCAL_DIR} ...", flush=True)
                    shutil.move(f, os.path.join(LOCAL_DIR, os.path.basename(f)))
                    last_file = os.path.basename(f)
                if glob.glob(os.path.join(LOCAL_DIR, d + "*.tif")):
                    moved.add(d)

        clear_output(wait=True)
        print("Earth Engine exports:", bar(len(completed) + len(failed), total))
        print(f"   completed {len(completed)} | running {states['RUNNING']} | queued {states['READY']} | "
              f"failed {len(failed)} | not found {total - len(seen)}")
        if move_files:
            print(f"Saved to {LOCAL_DIR}:", bar(len(moved), total))
            print("   last moved:", last_file)
        print(f"updated {time.strftime('%H:%M:%S')} - refreshing every {poll_seconds}s "
              "(stopping this cell does NOT cancel the exports)")

        exports_done = len(completed) + len(failed) >= total
        transfers_done = (not move_files) or completed <= moved
        if exports_done and transfers_done:
            break
        time.sleep(poll_seconds)

    for d in sorted(failed):
        print("PROBLEM:", d, "-", seen[d].status().get("error_message", seen[d].state))
    print("Finished." if not failed else f"Finished with {len(failed)} problem(s) listed above.")

run()

## 8. (Optional) Stack the downloaded months per county
Builds a lazy stack per county with a `time` dimension and named bands from the files in `F:\s2_bands`. Nothing is copied or loaded into memory until you compute, plot or save. No GDAL install needed.

In [ ]:
import re
import pandas as pd
import xarray as xr
import rioxarray

BAND_NAMES = BANDS + ["fill_flag"]      # order of bands inside each exported GeoTIFF

stacks = {}
for name in COUNTIES:
    files = sorted(glob.glob(os.path.join(LOCAL_DIR, f"S2_{name}_*.tif")))   # names sort chronologically
    if not files:
        print(f"No files found for {name}")
        continue

    times = [pd.Timestamp(int(y), int(m), 1)
             for y, m in (re.search(r"_(\d{4})_(\d{2})\.tif$", f).groups() for f in files)]

    das = [rioxarray.open_rasterio(f, chunks={"x": 1024, "y": 1024}) for f in files]
    da = xr.concat(das, dim=pd.Index(times, name="time"))
    da = da.assign_coords(band=BAND_NAMES)
    da.name = "S2"
    stacks[name] = da
    print(name, dict(da.sizes))

# Example (cast to float before band maths; the data are int16):
# da = stacks["Nyeri"].astype("float32")
# valid = da.sel(band="fill_flag") < 255
# ndvi = ((da.sel(band="B8") - da.sel(band="B4")) / (da.sel(band="B8") + da.sel(band="B4"))).where(valid)